In [1]:
import os

In [2]:
%pwd

'd:\\Chest-Cancer-Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Chest-Cancer-Classification'

In [5]:
import os

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/MSIVAPAPARAO13/Chest-Cancer-Classification.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "MSIVAPAPARAO13"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "82ff231b4fa55c88903b5616a24e4e9bab1108d1"

In [6]:
import tensorflow as tf

In [7]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

[2026-04-18 18:38:21,006] INFO | cnnClassifierLogger | Logging is configured successfully


In [11]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/Chest-CT-Scan-data",
            mlflow_uri="https://dagshub.com/MSIVAPAPARAO13/Chest-Cancer-Classification.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config




In [12]:
import pkg_resources
print("Working ✅")

Working ✅


In [13]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [14]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")


In [15]:
from cnnClassifier import logger

try:
    logger.info(" Evaluation stage started")

    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()

    evaluation = Evaluation(eval_config)

    evaluation.evaluation()
    evaluation.log_into_mlflow()

    logger.info(" Evaluation stage completed successfully")

except Exception as e:
    logger.exception(" Error in Evaluation stage")
    raise e

[2026-04-18 18:42:29,117] INFO | cnnClassifierLogger |  Evaluation stage started
[2026-04-18 18:42:29,164] INFO | cnnClassifierLogger | YAML loaded successfully: config\config.yaml
[2026-04-18 18:42:29,189] INFO | cnnClassifierLogger | YAML loaded successfully: params.yaml
[2026-04-18 18:42:29,195] INFO | cnnClassifierLogger | Directory created: artifacts
Found 102 images belonging to 2 classes.
7/7 [==============================] - 14s 2s/step - loss: 1.0508 - accuracy: 0.6078
[2026-04-18 18:42:46,202] INFO | cnnClassifierLogger | JSON saved at: scores.json
[2026-04-18 18:44:54,447] WARNING | urllib3.connectionpool | Retrying (Retry(total=4, connect=5, read=4, redirect=5, status=5)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='dagshub.com', port=443): Read timed out. (read timeout=120)")': /MSIVAPAPARAO13/Chest-Cancer-Classification.mlflow/api/2.0/mlflow/runs/create
[2026-04-18 18:46:59,006] WARNING | urllib3.connectionpool | Retrying (Retry(total=3, connec

2026/04/18 18:47:42 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-04-18 18:47:47,550] WARNING | absl | Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 14). These functions will not be directly callable after loading.
INFO:tensorflow:Assets written to: C:\Users\msiva\AppData\Local\Temp\tmp4vokzp8r\model\data\model\assets
[2026-04-18 18:47:52,257] INFO | tensorflow | Assets written to: C:\Users\msiva\AppData\Local\Temp\tmp4vokzp8r\model\data\model\assets


c:\Users\msiva\anaconda3\envs\cancer\lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
Successfully registered model 'VGG16Model'.


[2026-04-18 18:51:34,005] WARNING | urllib3.connectionpool | Retrying (Retry(total=4, connect=5, read=4, redirect=5, status=5)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='dagshub.com', port=443): Read timed out. (read timeout=120)")': /MSIVAPAPARAO13/Chest-Cancer-Classification.mlflow/api/2.0/mlflow/model-versions/create
[2026-04-18 18:52:04,475] WARNING | urllib3.connectionpool | Retrying (Retry(total=3, connect=5, read=4, redirect=5, status=5)) after connection broken by 'SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1017)'))': /MSIVAPAPARAO13/Chest-Cancer-Classification.mlflow/api/2.0/mlflow/model-versions/create


2026/04/18 18:52:08 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: VGG16Model, version 2
Created version '2' of model 'VGG16Model'.


[2026-04-18 18:52:10,092] INFO | cnnClassifierLogger |  Evaluation stage completed successfully


In [ ]:
from cnnClassifier import logger

try:
    logger.info(" Evaluation stage started")

    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()

    evaluation = Evaluation(eval_config)

    evaluation.evaluation()
    evaluation.log_into_mlflow()

    logger.info(" Evaluation stage completed successfully")

except Exception as e:
    logger.exception(" Error in Evaluation stage")
    raise e

[2026-04-17 15:55:01,922] INFO | cnnClassifierLogger | 🔍 Evaluation stage started
[2026-04-17 15:55:01,963] INFO | cnnClassifierLogger | YAML loaded successfully: config\config.yaml
[2026-04-17 15:55:01,975] INFO | cnnClassifierLogger | YAML loaded successfully: params.yaml
[2026-04-17 15:55:01,982] INFO | cnnClassifierLogger | Directory created: artifacts
Found 102 images belonging to 2 classes.
[2026-04-17 15:55:03,679] ERROR | cnnClassifierLogger | ❌ Error in Evaluation stage
Traceback (most recent call last):
  File "C:\Users\msiva\AppData\Local\Temp\ipykernel_23048\218604380.py", line 11, in <module>
    evaluation.evaluation()
  File "C:\Users\msiva\AppData\Local\Temp\ipykernel_23048\3217913975.py", line 39, in evaluation
    self.score = model.evaluate(self.valid_generator)
NameError: name 'model' is not defined


NameError: name 'model' is not defined

In [16]:
def evaluation(self):
    self.model = self.load_model(self.config.path_of_model)
    self._valid_generator()
    self.score = self.model.evaluate(self.valid_generator)  # FIXED
    self.save_score()